#### ENRICH GEO LOCATIONS FROM BRONZE TO SILVER

In [0]:
CATALOG = 'ORDERS_SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'GEO_LOCATIONS_ENRICHED'

In [0]:
geo_locations_df = spark.table('ORDERS_SALES.BRONZE.GEO_LOCATIONS')
geo_locations_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

geo_locations_trans_df = (geo_locations_df.select(
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_state",
    "geolocation_city",
    "file_name",
    "file_path",
    current_timestamp().alias('load_timestamp')
)
.withColumn("zip_prefix", col("geolocation_zip_code_prefix").cast("int"))
.withColumn("lat", col("geolocation_lng").cast("double"))
.withColumn("lng", col("geolocation_lng").cast("double"))
.withColumnRenamed("geolocation_state", "state")
.withColumnRenamed("geolocation_city", "city")
.drop("geolocation_lng", "geolocation_lat")
)

geo_locations_selected_df = geo_locations_trans_df.selectExpr("zip_prefix", "state", "city", "lat", "lng", "file_name", "file_path", "current_timestamp() as load_timestamp")
display(geo_locations_selected_df.limit(1))

In [0]:
geo_locations_selected_df.createOrReplaceTempView("geo_locations_selected_temp_vw")

In [0]:
%sql
SELECT * FROM geo_locations_selected_temp_vw limit 1

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.GEO_LOCATIONS_ENRICHED`

In [0]:
spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_customers_temp_vw
  )
  WHERE rn = 1
) src
ON (src.customer_id = tgt.customer_id)
WHEN MATCHED THEN
  UPDATE SET
    tgt.customer_id = src.customer_id,
    tgt.customer_unique_id = src.customer_unique_id,
    tgt.customer_zip_code_prefix = src.customer_zip_code_prefix,
    tgt.customer_city = src.customer_city,
    tgt.customer_state = src.customer_state
WHEN NOT MATCHED THEN
  INSERT (
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state,
    origin_file_path,
    load_timestamp
  )
  VALUES (
    src.customer_id,
    src.customer_unique_id,
    src.customer_zip_code_prefix,
    src.customer_city,
    src.customer_state,
    src.file_path,
    current_timestamp()
  )
""")

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.GEO_LOCATIONS_ENRICHED')